# Visualizing a single audio recording
### George Tzanetakis, University of Victoria

In this notebook we will explore some basic ways we can visualize audio time and frequency domain representations.


One easy possibility which we have already used is to create directly a line plot of the sample values.
However - the number of samples used is much higher than the actual values need for the plot so this approach
does not work on applications where drawing speed is important such as zooming in an audio editor. The clip shown is only 10 seconds but sometimes we need to visualize much longer audio recordings.

In [3]:
import numpy as np
import scipy.io.wavfile as wav
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
import IPython.display as ipd
import urllib.request



def load_audio(filename):
    srate, audio = wav.read(filename)
    # scale to -1.0/1.0
    audio = audio.astype(np.float32) / 32767.0

    # make max be 0.9
    if (len(audio.shape) == 1):
        audio = (0.9 / max(np.abs(audio)) * audio)
    else:
        audio[:,0] = (0.9 / max(np.abs(audio[:,0])) * audio[:,0])
        audio[:,1] = (0.9 / max(np.abs(audio[:,1])) * audio[:,1])
        return audio.transpose(), srate
    return audio, srate

url = "https://github.com/gtzan/mir_program_kadenze/raw/master/course2/session6/amazing_grace_female.wav"
filename = "amazing_grace_female.wav"
urllib.request.urlretrieve(url, filename)
(amazing_grace, srate) = load_audio(filename)

In [4]:
output_notebook()

def plot_samples(audio, srate):
    p = figure(width=800, height=200)
    time = np.linspace(0,len(audio)/srate, num=len(audio))
    p.line(time,amazing_grace)
    show(p)

plot_samples(amazing_grace, srate)
ipd.Audio(amazing_grace,rate=srate)

The most common way to visualize time-domain audio signals used by most audio editors and digital audio workstatios is a time-domain waveform. In this approach the audio signal is processed in blocks with their size and number determined by the zoom level. A single waveform point is calculated for each block. The most common choice is the maximum absolute value of the block sample amplitudes. For rendering vertical bars from -max(abs(block) to +max(abs(block) can be used or if alternatively we can connect them with a line alternating between the - and +values.

In [7]:
from bokeh.layouts import column


def max_absolute(block):
    return np.max(np.abs(block))

def process(signal,hopSize,block_process):
    winSize = hopSize
    offsets = np.arange(0, len(signal), hopSize)
    amp_track = np.zeros(len(offsets))
    for (m,o) in enumerate(offsets):
        frame = signal[o:o+winSize]
        amp_track[m] = block_process(np.abs(frame))
    return amp_track


def plot_envelope(audio,hopSize):
    max_envelope = process(audio,256,block_process=max_absolute)
    p = figure(width=800, height=200)
    time = np.linspace(0,len(audio)/srate, num=len(max_envelope))
    p.line(time,max_envelope)
    p.line(time,-max_envelope)
    return p

p1 = plot_envelope(amazing_grace, 256)

url = "https://github.com/gtzan/mir_program_kadenze/raw/master/course2/session6/amazing_grace_female.wav"
filename = "firebird3.wav"
urllib.request.urlretrieve(url, filename)
(firebird, srate) = load_audio('firebird3.wav')
p2 = plot_envelope(firebird, 256)

show(column([p1,p2]))

When the input audio is a stero file a common approach is to use the positive half of the plot for the left channel and the negative side of the plot for the right channel.

In [ ]:
(in_my_life,srate) = load_audio('in_my_life.wav')
display(ipd.Audio(in_my_life, rate=srate))


def plot_stereo_envelope(audio,hopSize):
    max_envelope_left = process(audio[0,:],256,block_process=max_absolute)
    max_envelope_right = process(audio[1,:], 256,block_process=max_absolute)
    p = figure(plot_width=800, plot_height=200)
    time = np.linspace(0,len(audio)/srate, num=len(max_envelope_left))
    p.line(time,max_envelope_right)
    p.line(time,-max_envelope_left)
    return p

p1 = plot_stereo_envelope(in_my_life, 256)
show(p1)

Another plot aspect that can be used to convey information about the underlying audio signal is color.
A typical approach when using color to display some kind of changing value is to utilize colormaps which map values to colors. Bokeh provides a variety of colormap options and it is possibly to design your own. Let's look at some examples in which we map the maximum absolute values to both the envelope and the color using different colormaps.

In [ ]:
from bokeh.palettes import Viridis256, Greys256, Magma256, Spectral11
from bokeh.layouts import column,row

def colored_waveform(times, envelope_values, colors, width=800, height=200):
    p = figure(plot_width=width, plot_height=height)
    p.vbar(x=times, bottom=-envelope_values, top=envelope_values,
           width=1.0/len(times),
           color=colors)
    return p

envelope_values = process(amazing_grace,256, max_absolute)
color_values = envelope_values
times = np.linspace(0,len(amazing_grace)/srate, num=len(envelope_values))

cplots = [colored_waveform(times,
                           envelope_values,
                           [cmap[int(v * (len(cmap)-1))] for v in color_values])
          for cmap in [Greys256, Spectral11, Viridis256, Magma256]]
show(column(cplots))

It also easy to make a color-bar plot that displays the audio using stripes of color disregarding the envelope information. In this case the amplitude information is just mapped to color.

In [ ]:
def colorbar(times, colors, width=800, height=200):
    p = figure(plot_width=width, plot_height=height)
    p.vbar(x=times, bottom=-1, top=1, width=1.0/len(times), color=colors)
    return p

color_values = process(amazing_grace,256, max_absolute)
times = np.linspace(0,len(amazing_grace)/srate, num=len(color_values))
cmap = Viridis256
colors = [cmap[int(v * (len(cmap)-1))] for v in color_values]

p = colorbar(times,colors)
show(p)

We can easily map other extracted features to color - for example here we map the spectral centroid.

In [ ]:
import scipy as sp


def spectral_centroid(block):
    fft = sp.fft.fft(block)
    spectrum = abs(fft)[:int(len(block)/2)]
    if (np.sum(spectrum) != 0.0):
            norm_spectrum = spectrum / np.sum(spectrum)
    else:
        norm_spectrum = spectrum
    fbins = np.arange(0.0,float(len(spectrum)))
    centroid = np.dot(fbins, norm_spectrum)
    return centroid

envelope_values = process(amazing_grace,256, max_absolute)
color_values = process(amazing_grace,256, spectral_centroid)
color_values /= np.max(color_values)
cmap = Spectral11
colors = [cmap[int(v * (len(cmap)-1))] for v in color_values]
times = np.linspace(0,len(amazing_grace)/srate, num=len(envelope_values))

p = colored_waveform(times,envelope_values,colors)
show(p)



In [ ]:
from scipy import signal
f, t, Sxx = signal.spectrogram(amazing_grace, srate)
# convert to dB
Sxx = 10 * np.log10(Sxx / np.max(Sxx))

p = figure(plot_width=800, plot_height=400, x_range=(0, t[-1]), y_range=(0, f[-1]))
p.grid.grid_line_width = 0.0

p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None

p.image(image=[Sxx], x=0, y=0, dw=t[-1], dh=f[-1], palette="Greys256", level="image")

show(p)

There are many cases where we are interested in visualizing multiple audio features. As we have seen in the first course of the program about extracting information from music signals when extracting audio features each block of audio is represented by a feature vector. So the entire recording can be viewed as a matrix where the x-axis is the block index and the y-axis is the feature number. Let's see how this works with the ubiquitious Mel-Frequency Cepstral Coefficients (MFCCs).  

In [ ]:


from sklearn.preprocessing import StandardScaler,MinMaxScaler
import librosa

def extract_mfcc(audio,srate,hopSize):
    mfcc_matrix = librosa.feature.mfcc(y=audio, sr=srate, hop_length=hopSize, dct_type=3)
    sc = StandardScaler()
    scaled_mfcc_matrix = sc.fit_transform(mfcc_matrix)
    return scaled_mfcc_matrix


mfcc_matrix = extract_mfcc(amazing_grace, srate, 256)

In [ ]:
p = figure(plot_width=800, plot_height=400, x_range=(0, t[-1]), y_range=(0, mfcc_matrix.shape[0]))
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None
p.image(image=[mfcc_matrix], x=0, y=0, dw=t[-1], dh=mfcc_matrix.shape[0], palette="Greys256", level="image")
show(p)

Another possibility is to use a dimensionality reduction techniques (such as Principal Component Analysis PCA) to reduce the 20-dimensional MFCC vectors to fewer dimensions for visualization. PCA transforms the input feature vectors to a lower dimensional vector representation that captures as much as possible of the variance in the data. If we use components for the PCA then we will get a sequence of 3-dimensional vectors that after normalization can be mapped to RGB color values. Let's see how this works. As the MFCCs capture timbral information these visualizations have been called timbregrams.

In [ ]:

from sklearn.decomposition import PCA
from bokeh.colors import RGB


def matrix2colors(train_matrix, transform_matrix):
    pca = PCA(n_components=3)
    pca.fit(train_matrix.transpose())
    pca_mfcc = pca.transform(transform_matrix.transpose())

    explained_variance = pca.explained_variance_ratio_
    print('Explained variance of components', explained_variance)
    print('Total explained variance', np.sum(explained_variance))

    min_max = MinMaxScaler()
    pca_mfcc = min_max.fit_transform(pca_mfcc) * 255
    mfcc_colors = pca_mfcc.astype(int)
    mfcc_colors = [RGB(r,g,b) for [r,g,b] in mfcc_colors]
    return mfcc_colors




In [ ]:
def timbregram(audio,train_matrix,transform_matrix, width=800, height=200, waveform=True):
    colors = matrix2colors(train_matrix, transform_matrix)
    envelope_values = process(audio,256, max_absolute)
    times = np.linspace(0,len(audio)/srate, num=len(envelope_values))
    if waveform:
        p = colored_waveform(times,envelope_values,colors,width, height)
    else:
        p = colorbar(times,colors, width, height)
    return p

(f4,srate) = load_audio('firebird4.wav')

mfcc_matrix = extract_mfcc(f4, srate, 256)
p = timbregram(f4,mfcc_matrix,mfcc_matrix)
show(p)

Explained variance of components [0.6292506  0.11562341 0.05660553]
Total explained variance 0.8014795


PCA tries to explain the variance over a dataset. Once the PCA transformation it can be applied to new data.
This opens up the potential for context-aware visualizations in which the coloring of an audio recording does not only depend on the underlying content but also how it relates to other recordings in a larger collection.
Let's compute MFCCs over a small collections of three 30-second excerpts of a performance of the Firebird by Stravinsky. We can train a PCA transformation using the MFCCs from all the recordings. Notice that the resulting mapping to colors of the same audio recording is different as the PCA transform explains the variance over the collection rather than the individual recording. We will see how this can be used to show timbral similarity and music structure in a visual way.

In [ ]:
audio_recordings = [load_audio(fname)[0] for fname in ['firebird1.wav','firebird3.wav','firebird4.wav']]
mfcc_matrices = [extract_mfcc(audio, srate, 256) for audio in audio_recordings]
fcollection = np.hstack(mfcc_matrices)
p = timbregram(f4, fcollection, mfcc_matrix, 800,200)
show(p)

Explained variance of components [0.5352213  0.22443561 0.04660772]
Total explained variance 0.80626464


We can plot colored waveforms and bars for all the pieces in the small collection. Notice that one can see that the third piece has two clearly different parts and that the first part looks similar in terms of color with the first piece. This information is obtained immediately without having to hear the actual recording and shows the advantage of a content and context aware visualization.

In [ ]:
mfcc_matrices = [extract_mfcc(audio, srate, 256) for audio in audio_recordings]
fcollection = np.hstack(mfcc_matrices)

plots1 = [timbregram(a, fcollection, m, 400,100) for (a,m) in zip(audio_recordings,mfcc_matrices)]
plots2 = [timbregram(a, fcollection, m, 400,100, waveform=False) for (a,m) in zip(audio_recordings,mfcc_matrices)]

show(row([column(plots1), column(plots2)]))

Explained variance of components [0.53522176 0.22443551 0.04660774]
Total explained variance 0.806265
Explained variance of components [0.53522146 0.22443557 0.04660777]
Total explained variance 0.8062648
Explained variance of components [0.53522176 0.22443557 0.04660774]
Total explained variance 0.80626506
Explained variance of components [0.53522146 0.22443576 0.04660771]
Total explained variance 0.80626494
Explained variance of components [0.535221   0.22443573 0.04660769]
Total explained variance 0.8062644
Explained variance of components [0.53522164 0.22443561 0.04660773]
Total explained variance 0.806265


Let's listen to the first and last recording and see if we can hear the two sections and the similarity of the first section of the third recording to the first recording.

In [ ]:
display(ipd.Audio(audio_recordings[0],rate=srate))
display(ipd.Audio(audio_recordings[2],rate=srate))

Notice that the sectioning of the third recording which is clearly visible with the MFCCs mapped to PCA transformed colors is barely noticeable when we visualize the same three recordings mapping the spectral centroid to color.

In [ ]:

def magma_centroid(audio, width, height):
    envelope_values = process(audio,256, max_absolute)
    color_values = process(audio,256,spectral_centroid)
    color_values /= np.max(color_values)
    times = np.linspace(0,len(amazing_grace)/srate, num=len(envelope_values))
    cmap = Magma256
    p = colored_waveform(times,
                        envelope_values,
                        [cmap[int(v * (len(cmap)-1))] for v in color_values], width, height)
    return p


plots = [magma_centroid(a, 500,150) for a in audio_recordings]
show(column(plots))